In [ ]:
# ============================================================ #
# Cell 0: Download resume state từ Google Drive (chỉ chạy khi resume)
# ============================================================ #
# Đặt TRAIN_PHASE ở đây:
#   1  -> train task 0-1, lưu continuation_state_task_1.pt
#   2  -> load task_1 state, train task 2, lưu continuation_state_task_2.pt
#   3  -> load task_2 state, train task 3, lưu continuation_state_task_3.pt
#   4  -> load task_3 state, train task 4-5 (xong)
#   5  -> train toàn bộ task 0-5 (không resume)
# ============================================================ #
import os
import subprocess
import sys
import zipfile

TRAIN_PHASE = 5  # <<< SỬA Ở ĐÂY: 1=task0-1, 2=task2, 3=task3, 4=task4-5, 5=all
TRAIN_SEED = 42  # P6: chạy lần lượt 42, 43, 44; mỗi seed có output riêng
TRAIN_OUTPUT_DIR = f'/kaggle/working/results_denice_seed_{TRAIN_SEED}'

PHASE_CONFIG = {
    1: {
        "task_start": 0,
        "task_end": 1,
        "save_resume_after_task": 1,
        "resume_file": None,
    },
    2: {
        "task_start": 2,
        "task_end": 2,
        "save_resume_after_task": 2,
        "resume_file": "continuation_state_task_1.pt",
    },
    3: {
        "task_start": 3,
        "task_end": 3,
        "save_resume_after_task": 3,
        "resume_file": "continuation_state_task_2.pt",
    },
    4: {
        "task_start": 4,
        "task_end": 5,
        "save_resume_after_task": None,
        "resume_file": "continuation_state_task_3.pt",
    },
    5: {
        "task_start": 0,
        "task_end": 5,
        "save_resume_after_task": None,
        "resume_file": None,
    },
}

phase_config = PHASE_CONFIG[TRAIN_PHASE]
desired_resume_file = phase_config["resume_file"]
target = None

if desired_resume_file is None:
    print(f"Phase {TRAIN_PHASE}: không cần resume state.")
else:
    os.makedirs("/tmp/next/continue", exist_ok=True)

    # <<< ĐỔI URL NÀY thành link Google Drive chứa file .zip resume state >>>
    GDRIVE_URL = "https://drive.google.com/file/d/1OGvIpqrWJ1fdtwZpttq4OogZC7m8uBKo/view?usp=sharing"

    archive_path = "/tmp/next/continue/continue.zip"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run(
        [sys.executable, "-m", "gdown", "--fuzzy", GDRIVE_URL, "-O", archive_path],
        check=True,
    )

    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall("/tmp/next/")

    resume_candidates = []
    for root, _, files in os.walk("/tmp/next"):
        for f in files:
            if f.endswith(".pt"):
                resume_candidates.append(os.path.join(root, f))

    print("PT files found:")
    for p in resume_candidates:
        print(p)

    for p in resume_candidates:
        if os.path.basename(p).lower() == desired_resume_file:
            target = p
            break

    if target is None:
        # Fallback: older result zips may contain checkpoint_task_N.pt only,
        # not continuation_state_task_N.pt. Wrap checkpoint into a minimal
        # continuation payload so split training can resume from task N+1.
        import torch

        completed_task_id = int(phase_config["task_start"]) - 1
        fallback_name = f"checkpoint_task_{completed_task_id}.pt"
        fallback_path = None
        for p in resume_candidates:
            if os.path.basename(p).lower() == fallback_name.lower():
                fallback_path = p
                break

        if fallback_path is None:
            raise FileNotFoundError(
                f"Phase {TRAIN_PHASE} yêu cầu {desired_resume_file} hoặc {fallback_name} "
                "nhưng không tìm thấy trong /tmp/next."
            )

        ckpt = torch.load(fallback_path, map_location="cpu", weights_only=False)
        ckpt_algorithm = ckpt.get("algorithm") or ckpt.get("config", {}).get("algorithm")
        ckpt_mode = ckpt.get("mode") or ckpt.get("config", {}).get("mode", "il")
        ckpt_completed_task = int(ckpt.get("task_id", completed_task_id))
        task_accuracies = ckpt.get("task_accuracies") or []
        if not isinstance(task_accuracies, list) or not task_accuracies:
            task_accuracies = [
                {"task": tid, "accuracy": None}
                for tid in range(ckpt_completed_task + 1)
            ]

        continuation_state = {
            "meta": {
                "mode": ckpt_mode,
                "algorithm": ckpt_algorithm,
                "completed_task": ckpt_completed_task,
                "resume_from_task": ckpt_completed_task + 1,
                "output_dir": os.path.dirname(fallback_path),
                "source_checkpoint": fallback_path,
            },
            "config": ckpt.get("config", {}),
            "model_state_dict": ckpt["model_state_dict"],
            "global_neuron_ages": ckpt.get("global_neuron_ages"),
            "all_history": {
                "task_accuracies": task_accuracies,
                "task_forgetting": [],
                "round_metrics": [],
            },
            "best_acc_per_task": {},
            "seen_classes": ckpt.get("seen_classes", []),
            "trainer_state": None,
            "server_state": None,
            "aggregator_state": None,
            "persistent_clients_state": {},
        }
        target = os.path.join("/tmp/next", desired_resume_file)
        torch.save(continuation_state, target)
        print("Wrapped checkpoint resume state:", target)
        print("Source checkpoint:", fallback_path)

print("Selected resume path:", target)
if target is not None:
    print("exists:", os.path.exists(target))
    print("size:", os.path.getsize(target))

In [ ]:
"""
Federated Class Incremental Learning - Training Entry Point (Kaggle)
=====================================================================
Chỉnh CONFIG bên dưới rồi Run All. Cell 0 đã set TRAIN_PHASE và `target`.
"""

import os
import sys

# =============================================================================
# KAGGLE SETUP - Clone from GitHub (luôn fresh để lấy code mới nhất)
# =============================================================================
REPO_PATH = "/tmp/FL_IL_IDS"


def setup_imports():
    import shutil
    if os.path.exists(REPO_PATH):
        print(f"Removing stale clone at {REPO_PATH}...")
        shutil.rmtree(REPO_PATH)
    print("Cloning from GitHub...")
    os.system(f"git clone https://github.com/khoilv2005/FL_IL_IDS.git {REPO_PATH}")
    new_sys_path = [p for p in sys.path if not p.startswith("/kaggle/input")]
    sys.path = [REPO_PATH] + new_sys_path
    for k in [k for k in sys.modules if "fed_learning" in k]:
        del sys.modules[k]
    print(f"sys.path[0]: {sys.path[0]}")


setup_imports()


# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # ------------------------------------------------------------------
    # Data
    # ------------------------------------------------------------------
    "data_dir": "/kaggle/input/datasets/khoilv2005/100-clients/100-clients",
    "random_seed": TRAIN_SEED,

    # ------------------------------------------------------------------
    # Training mode + algorithm
    # mode:      "fed_il" | "il" | "decentralized"
    # algorithm: fed_il -> "cgofed" | "fedavg_ewc" | "fedprox_ewc" | "fedavg_lwf"
    #                      "fedprox_lwf" | "fedcbdr" | "der" | "nice" | "glfc"
    #                      "refed" | "dfca_il"
    #            decentralized -> "plexus" | "denice"
    #            il            -> "ewc" | "lwf" | "der" | "nice" | "denice"
    # ------------------------------------------------------------------
    "mode": "decentralized",
    "algorithm": "denice",

    # ------------------------------------------------------------------
    # Output — /kaggle/working/ tự động download được từ tab Output
    # ------------------------------------------------------------------
    "output_dir": TRAIN_OUTPUT_DIR,

    # ------------------------------------------------------------------
    # Split-run / continuation — lấy từ Cell 0 (phase_config + target)
    # ------------------------------------------------------------------
    "task_start": phase_config["task_start"],
    "task_end":   phase_config["task_end"],
    "save_resume_after_task": phase_config["save_resume_after_task"],
    "resume_state_path": target,          # None nếu Phase 5 / không resume
    "resume_output_dir": TRAIN_OUTPUT_DIR,

    # ------------------------------------------------------------------
    # Incremental learning setup — 6 tasks × (6,6,6,6,6,4) = 34 classes
    # ------------------------------------------------------------------
    "num_clients": 100,
    "total_classes": 34,
    "base_classes": 6,
    "classes_per_task": 6,

    # ------------------------------------------------------------------
    # Common hyper-parameters
    # ------------------------------------------------------------------
    "mu_fedprox": 0.0,
    "rounds_per_task": 20,
    "local_epochs": 1,
    "learning_rate": 0.001,
    "batch_size": 2048,
    "eval_batch_size": 8192,
    # eval_every > rounds_per_task -> bỏ eval giữa round.
    # Chỉ chạy quick diagnostic ở task 5 / round 19; E0-E6 chi tiết ở Cell 2.
    "eval_every": 9999,
    "round_checkpoint_every": 1,

    # ------------------------------------------------------------------
    # CGoFed
    # ------------------------------------------------------------------
    "mu_cgofed": 1.0,
    "lambda_decay": 0.8,
    "theta_threshold": 0.35,
    "cross_task_weight": 0.3,
    "lambda_cross_task": 0.3,
    "energy_threshold": 0.99,
    "num_samples_rep": 1000,
    "top_k": 2,

    # EWC
    "ewc_lambda": 1000.0,
    "fisher_samples": 200,
    "online_ewc": False,

    # LwF
    "lwf_alpha": 1.0,
    "temperature": 2.0,
    "lwf_alpha_scale": 1.0,
    "distill_old_classes_only": False,

    # FedCBDR
    "tau_old": 0.9,
    "tau_new": 1.1,
    "omega_old": 1.1,
    "omega_new": 0.9,
    "buffer_size": 500,
    "replay_ratio": 0.5,
    "rne_feature_batch_size": 1024,
    "seed": TRAIN_SEED,

    # DER
    "lambda_aux": 1.0,
    "lambda_sparsity": 0.1,
    "s_max": 15.0,
    "der_temperature": 2.0,
    "der_stage1_rounds": 12,
    "der_stage2_rounds": 8,

    # ------------------------------------------------------------------
    # NICE / DeNICE
    # ------------------------------------------------------------------
    "tau": 0.95,
    "nice_max_phases": 20,
    "nice_phase_epochs": 1,
    "memo_per_class": 50,
    "nice_context_eval": True,
    "nice_debug_context_detector": True,

    # DeNICE adapter layers
    # ["fc1"]                 -> Phase 1 MVP (nhẹ nhất)
    # ["fc1", "gru"]          -> Phase 2a
    # ["fc1", "gru", "conv3"] -> Phase 2b (đầy đủ)
    "denice_adapter_layers": ["fc1", "gru", "conv3"],
    "denice_debug": True,
    "denice_save_round_artifacts": False,
    "denice_checkpoint_format": "delta",

    # Quick final-round metrics: balanced subset + clients có đủ episode coverage.
    "denice_post_task_eval": True,
    "denice_post_task_eval_tasks": [5],
    "denice_eval_max_clients": 3,
    "denice_eval_require_full_coverage": True,
    "denice_eval_max_samples": 50000,
    "denice_eval_progress_every_clients": 10,  # in progress mỗi 10 clients
    "denice_eval_progress_every_batches": 0,
    "denice_eval_route_mode": "hard",
    "denice_eval_route_topk": 1,
    "denice_eval_report_nomask": True,
    "denice_eval_representative_ensemble": True,

    # DeNICE context routing bank. scope="cluster" follows the proposal:
    # share context capsule/sketches only inside decentralized collaboration group.
    # Use scope="global" only for ablation/debug.
    "denice_shared_context_eval": True,
    "denice_shared_context_scope": "cluster",
    "denice_shared_context_max_per_episode": 512,
    "denice_shared_context_require_compatible_calibration": True,
    "denice_router_mode": "multiclass",
    "denice_refresh_router_memory_after_aggregation": True,
    "denice_router_update_schedule": "task_end",

    # DeNICE decentralized aggregation (Đề xuất §6-§7). Mặc định giữ hành vi cũ.
    # denice_aggregation_method: "weighted_mean" (mặc định, alpha_ij chuẩn)
    #                            | "coordinate_median" | "trimmed_mean" (robust §7)
    "denice_aggregation_method": "weighted_mean",
    "denice_aggregation_trim_ratio": 0.1,
    "denice_aggregation_count_transform": "log",
    "denice_aggregation_self_floor": 0.25,
    "denice_gamma": 0.15,
    # G_i = {j | cùng cluster AND s_ij > delta} (§6). True = lọc theo đồ thị context.
    "denice_collab_use_context_edges": True,
    "denice_require_label_overlap": True,
    "denice_centroid_gate_threshold": 0.75,
    "denice_cluster_delta_sim": 0.0,  # <=0 dùng adaptive threshold
    "denice_cluster_edge_top_k": 40,
    "denice_cluster_edge_quantile": 0.25,
    "denice_cluster_min_signal_std": 0.02,
    "denice_cluster_theta_s": 0.5,
    "denice_cluster_invalid_policy": "previous_valid_or_self_only",
    "denice_age_merge_policy": "consensus",
    "denice_age_merge_consensus_threshold": 0.5,
    "denice_min_free_capacity_ratio": 0.10,

    # DeNICE graceful recycling
    "denice_enable_recycling": False,
    "denice_recycle_ratio": 0.02,
    "denice_recycle_min": 1,
    "denice_recycle_max_per_layer": 8,
    "denice_recycle_grace_tasks": 1,
    "denice_recycle_usage_recent_threshold": 0.10,
    "denice_recycle_max_old_metric_drop": 0.02,
    "denice_recycle_require_old_check": True,

    # ------------------------------------------------------------------
    # GLFC
    # ------------------------------------------------------------------
    "glfc_memory_size": 2000,
    "glfc_entropy_threshold": 1.2,
    "glfc_distill_weight": 0.5,
    "glfc_recon_iters": 250,
    "glfc_num_recon_images": 20,

    # Re-Fed
    "refed_memory_size": 2000,
    "refed_lambda_pim": 0.5,
    "refed_pim_iterations": 5,

    # ------------------------------------------------------------------
    # Plexus
    # ------------------------------------------------------------------
    "plexus_sample_size": 10,
    "plexus_num_aggregators": 1,
    "plexus_success_fraction": 0.8,
    "plexus_inactivity_threshold": 50,
    "plexus_scale_clients": True,
    "plexus_initial_client_ratio": 0.5,
    "plexus_final_client_ratio": 1.0,
}


# =============================================================================
# MAIN — chạy trực tiếp (không cần if __name__ == "__main__" trong Jupyter)
# =============================================================================
from fed_learning.training.task_loop import run_incremental_training

run_incremental_training(CONFIG)

In [ ]:
# =============================================================================
# EVALUATION — checkpoint cuối: task 5, round 19 (cumulative task 0-5)
# Chạy cell này sau khi cell train hoàn tất.
# =============================================================================
from pathlib import Path
import json
import subprocess
import sys
import torch

EVAL_MAX_SAMPLES = 50_000  # Fast debug subset; use None for full test set.
EVAL_SEED = TRAIN_SEED
checkpoint_path = Path(TRAIN_OUTPUT_DIR) / 'checkpoint_task_5_round_19.pt'
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Final checkpoint not found: {checkpoint_path}')

DEBUG_DIR = Path(TRAIN_OUTPUT_DIR) / 'final_round19_debug'
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
eval_device = 'cuda' if torch.cuda.is_available() else 'cpu'
common = [
    sys.executable, str(Path(REPO_PATH) / 'eval_checkpoint.py'),
    '--checkpoint', str(checkpoint_path), '--data-dir', str(CONFIG['data_dir']),
    '--device', eval_device, '--router-mode', 'multiclass',
    '--eval-seed', str(EVAL_SEED),
]
if EVAL_MAX_SAMPLES is not None:
    common += ['--max-samples', str(EVAL_MAX_SAMPLES)]

print(f'Checkpoint: {checkpoint_path}')
print(f'Debug output: {DEBUG_DIR} | device={eval_device} | max_samples={EVAL_MAX_SAMPLES}')

# Router sanity: stored memory and final-model current-feature routing.
router_memory_path = DEBUG_DIR / 'router_memory_audit.json'
router_feature_path = DEBUG_DIR / 'router_current_feature_audit.json'
subprocess.run([*common, '--router-audit', '--output', str(router_memory_path)], check=True)
subprocess.run([*common, '--router-current-feature-audit', '--router-audit-max-clients', '10', '--router-audit-samples-per-episode', '256', '--output', str(router_feature_path)], check=True)

# E0-E6 use the same checkpoint, seed, and quick test subset.
POLICIES = {
    'e0_backbone_nomask': ['--inference-policy', 'backbone_nomask'],
    'e1_pred_adapter_nomask': ['--inference-policy', 'pred_adapter_nomask'],
    'e2_oracle_adapter_nomask': ['--inference-policy', 'oracle_adapter_nomask'],
    'e3_oracle_hard': ['--inference-policy', 'oracle_hard'],
    'e4_pred_hard': ['--inference-policy', 'pred_hard'],
    'e5_topk2': ['--route-mode', 'topk', '--route-topk', '2'],
    'e6_adaptive': ['--route-mode', 'adaptive', '--route-topk', '2'],
}
results = {}
for name, policy_args in POLICIES.items():
    path = DEBUG_DIR / f'{name}.json'
    subprocess.run([*common, '--evaluation-mode', 'coverage_aware_local', *policy_args, '--output', str(path)], check=True)
    results[name] = json.loads(path.read_text(encoding='utf-8'))

print('\n=== FINAL ROUND 19 DEBUG SUMMARY ===')
for name, result in results.items():
    m = result['metrics']
    print(f"{name:25s} acc={m.get('accuracy', 0):.2%} macro_f1={m.get('f1_macro', 0):.2%} precision={m.get('precision_macro', 0):.2%} recall={m.get('recall_macro', 0):.2%} route_acc={m.get('route_accuracy', 0):.2%} route_coverage={m.get('route_coverage', 0):.2%} active_adapter={m.get('adapter_active_sample_count', 0)} missing_adapter={m.get('adapter_missing_sample_count', 0)} oracle_violations={m.get('oracle_mask_violation_count', 0)}")

memory_audit = json.loads(router_memory_path.read_text(encoding='utf-8'))
feature_audit = json.loads(router_feature_path.read_text(encoding='utf-8'))
print('\nRouter memory audit:', json.dumps(memory_audit.get('summary', memory_audit), indent=2))
print('Router current-feature audit:', json.dumps(feature_audit.get('summary', feature_audit), indent=2))

summary_path = DEBUG_DIR / 'final_round19_debug_summary.json'
summary_path.write_text(json.dumps({'checkpoint': str(checkpoint_path), 'seed': EVAL_SEED, 'max_samples': EVAL_MAX_SAMPLES, 'evaluation_mode': 'coverage_aware_local', 'router_memory_audit': str(router_memory_path), 'router_current_feature_audit': str(router_feature_path), 'results': results}, indent=2), encoding='utf-8')
print(f'\nSaved all metrics and debug logs to: {DEBUG_DIR}')
print(f'Summary file: {summary_path}')
